# Student Internship Placement Prediction using MLP
**Dataset:** Student Placement Prediction Dataset 2026 (Kaggle — sehaj1104, 100,000 records)  
**Model:** Multi-Layer Perceptron (MLP) Classifier  
**Goal:** Predict whether a student will be placed based on academic & professional attributes

---
## Phase 1 — Load & Explore Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, os, time

warnings.filterwarnings('ignore')
os.makedirs('../results', exist_ok=True)
os.makedirs('../models', exist_ok=True)

DATA_PATH = '../dataset/student_placement_prediction_dataset_2026.csv'
df = pd.read_csv(DATA_PATH)

print('=== Dataset Overview ===')
print(f'Records  : {df.shape[0]:,}')
print(f'Features : {df.shape[1]}')
print(f'Missing  : {df.isnull().sum().sum()}')
print()
print('=== Placement Distribution ===')
vc  = df['placement_status'].value_counts()
pct = df['placement_status'].value_counts(normalize=True).mul(100).round(1)
for label in vc.index:
    print(f'  {label:12s}: {vc[label]:>7,} ({pct[label]:.1f}%)')
df.head()

In [ ]:
print('=== Column Types & Ranges ===')
df.info()
print()
print('=== Numeric Summary ===')
df.describe().round(2)

In [ ]:
# ── EDA Plots ─────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Exploratory Data Analysis — Student Placement Dataset 2026',
             fontsize=14, fontweight='bold')
colors = {'Placed': '#2ecc71', 'Not Placed': '#e74c3c'}

# 1. Placement count
vc_vals = df['placement_status'].value_counts()
axes[0,0].bar(vc_vals.index, vc_vals.values,
              color=[colors[k] for k in vc_vals.index], edgecolor='black', linewidth=0.8)
axes[0,0].set_title('Placement Status Distribution'); axes[0,0].set_ylabel('Count')
for i, v in enumerate(vc_vals.values):
    axes[0,0].text(i, v + 300, f'{v:,}', ha='center', fontweight='bold')

# 2. CGPA by placement
for status, color in colors.items():
    axes[0,1].hist(df[df['placement_status']==status]['cgpa'],
                   bins=40, alpha=0.65, label=status, color=color, edgecolor='black', linewidth=0.3)
axes[0,1].set_title('CGPA Distribution by Placement Status')
axes[0,1].set_xlabel('CGPA'); axes[0,1].set_ylabel('Frequency'); axes[0,1].legend()

# 3. Placement rate by College Tier
tier_rate = df.groupby('college_tier')['placement_status'].apply(
    lambda x: (x=='Placed').mean() * 100).reset_index()
tier_rate.columns = ['college_tier', 'placement_rate']
axes[1,0].bar(tier_rate['college_tier'], tier_rate['placement_rate'],
              color=['#3498db','#9b59b6','#e67e22'], edgecolor='black')
axes[1,0].set_title('Placement Rate by College Tier')
axes[1,0].set_xlabel('College Tier'); axes[1,0].set_ylabel('Placement Rate (%)')
axes[1,0].set_ylim(0, 100)
for _, row in tier_rate.iterrows():
    axes[1,0].text(row['college_tier'], row['placement_rate']+1,
                   f"{row['placement_rate']:.1f}%", ha='center', fontweight='bold')

# 4. Coding skill by placement
for status, color in colors.items():
    axes[1,1].hist(df[df['placement_status']==status]['coding_skill_score'],
                   bins=40, alpha=0.65, label=status, color=color, edgecolor='black', linewidth=0.3)
axes[1,1].set_title('Coding Skill Score by Placement Status')
axes[1,1].set_xlabel('Coding Skill Score'); axes[1,1].set_ylabel('Frequency'); axes[1,1].legend()

plt.tight_layout()
plt.savefig('../results/eda_plots.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → results/eda_plots.png')

---
## Phase 1 — Data Preprocessing

In [ ]:
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.model_selection import train_test_split

df_model = df.copy()

# ── Step 1: Drop irrelevant / leakage columns ─────────────────────────────────
df_model.drop(columns=['student_id', 'salary_package_lpa'], inplace=True)
print('Dropped: student_id (irrelevant), salary_package_lpa (data leakage)')

# ── Step 2: Encode TARGET ─────────────────────────────────────────────────────
df_model['placement_status'] = (df_model['placement_status'] == 'Placed').astype(int)
print('Target → Placed=1, Not Placed=0')
print('Target distribution:', df_model['placement_status'].value_counts().to_dict())

# ── Step 3: Binary encode simple 2-class categoricals ────────────────────────
df_model['gender'] = df_model['gender'].map({'Male': 1, 'Female': 0})
df_model['volunteer_experience'] = df_model['volunteer_experience'].map({'Yes': 1, 'No': 0})
print('Binary encoded: gender (Male=1), volunteer_experience (Yes=1)')

# ── Step 4: Ordinal encode college_tier (Tier1 best=2, Tier3 worst=0) ─────────
oe = OrdinalEncoder(categories=[['Tier 3', 'Tier 2', 'Tier 1']])
df_model['college_tier'] = oe.fit_transform(df_model[['college_tier']]).astype(int)
print('Ordinal encoded: college_tier (Tier 1=2, Tier 2=1, Tier 3=0)')

# ── Step 5: One-Hot Encode branch (multi-class nominal) ───────────────────────
print('Branches found:', sorted(df_model['branch'].unique()))
df_model = pd.get_dummies(df_model, columns=['branch'], prefix='branch')
print('One-hot encoded branch. New branch columns:', [c for c in df_model.columns if c.startswith('branch_')])

print(f'\nFinal shape: {df_model.shape}')
df_model.head(3)

In [ ]:
# ── Feature correlation analysis ──────────────────────────────────────────────
print('=== Feature Correlations with placement_status (all models) ===')
corr = df_model.corr()['placement_status'].drop('placement_status').abs().sort_values(ascending=False)
print(corr.round(4).to_string())
print(f'\n→ Max |correlation| = {corr.max():.4f} ({corr.idxmax()})')
print('→ This reveals a weakly-correlated (synthetic) dataset — accuracy ceiling ~57%')

In [ ]:
# ── Feature / Target split ────────────────────────────────────────────────────
X = df_model.drop('placement_status', axis=1)
y = df_model['placement_status']
feature_names = list(X.columns)

print(f'Total features: {len(feature_names)}')
for i, f in enumerate(feature_names, 1):
    print(f'  {i:2d}. {f}')

# ── CORRECT ORDER: Split FIRST, then scale ────────────────────────────────────
# (fit scaler on train only — prevents test data leakage)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)   # fit on train
X_test_sc  = scaler.transform(X_test)        # transform test with same params

print(f'\nTrain: {X_train_sc.shape[0]:,} | Test: {X_test_sc.shape[0]:,}')
print('Scaler fitted on training set only (no test leakage)')

---
## Phase 1 — Train Baseline MLP (64, 32)

In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import joblib

print('Training Baseline MLP — hidden_layer_sizes=(64, 32)...')
t0 = time.time()
mlp_baseline = MLPClassifier(
    hidden_layer_sizes=(64, 32),
    activation='relu',
    solver='adam',
    learning_rate='adaptive',
    max_iter=500,
    random_state=42,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=20,
    verbose=False
)
mlp_baseline.fit(X_train_sc, y_train)

y_pred_baseline = mlp_baseline.predict(X_test_sc)
acc_baseline    = accuracy_score(y_test, y_pred_baseline)

print(f'Time     : {time.time()-t0:.1f}s  | Iterations: {mlp_baseline.n_iter_}')
print(f'Accuracy : {acc_baseline*100:.2f}%')
print()
print(classification_report(y_test, y_pred_baseline, target_names=['Not Placed', 'Placed']))

In [ ]:
# ── Confusion Matrix ───────────────────────────────────────────────────────────
cm = confusion_matrix(y_test, y_pred_baseline)

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Not Placed', 'Placed'],
            yticklabels=['Not Placed', 'Placed'],
            linewidths=0.5, linecolor='gray', ax=ax,
            annot_kws={'size': 14, 'weight': 'bold'})
ax.set_title(f'Confusion Matrix — Baseline MLP (64, 32)\nAccuracy: {acc_baseline*100:.2f}%',
             fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Predicted Label', fontsize=11)
ax.set_ylabel('True Label', fontsize=11)
plt.tight_layout()
plt.savefig('../results/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → results/confusion_matrix.png')

---
## Phase 2 — Architecture Comparison (3 MLP Experiments)

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

experiments = [
    {'name': 'MLP-1', 'label': '(32,)',        'hidden_layer_sizes': (32,)},
    {'name': 'MLP-2', 'label': '(64, 32)',     'hidden_layer_sizes': (64, 32)},
    {'name': 'MLP-3', 'label': '(128, 64, 32)','hidden_layer_sizes': (128, 64, 32)},
]

results_list   = []
trained_models = {}

print(f"{'Model':<8} {'Architecture':<16} {'Acc':>8} {'Prec':>8} {'Rec':>8} {'F1':>8} {'Iters':>7}")
print('-' * 65)

for exp in experiments:
    t0  = time.time()
    clf = MLPClassifier(
        hidden_layer_sizes=exp['hidden_layer_sizes'],
        activation='relu', solver='adam',
        learning_rate='adaptive',
        max_iter=500, random_state=42,
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=20
    )
    clf.fit(X_train_sc, y_train)
    y_pred = clf.predict(X_test_sc)

    acc  = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='weighted')
    rec  = recall_score(y_test, y_pred, average='weighted')
    f1   = f1_score(y_test, y_pred, average='weighted')

    results_list.append({
        'Model': exp['name'], 'Architecture': exp['label'],
        'Accuracy (%)': round(acc*100, 2), 'Precision (%)': round(prec*100, 2),
        'Recall (%)': round(rec*100, 2), 'F1 Score (%)': round(f1*100, 2),
        'Iterations': clf.n_iter_
    })
    trained_models[exp['name']] = clf
    print(f"{exp['name']:<8} {exp['label']:<16} {acc*100:>7.2f}% {prec*100:>7.2f}% {rec*100:>7.2f}% {f1*100:>7.2f}% {clf.n_iter_:>7d}  ({time.time()-t0:.1f}s)")

results_df = pd.DataFrame(results_list)
print()
print(results_df.to_string(index=False))

In [ ]:
# ── Architecture comparison charts ────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('MLP Architecture Comparison', fontsize=14, fontweight='bold')
bar_colors = ['#3498db', '#2ecc71', '#9b59b6']
metrics    = ['Accuracy (%)', 'Precision (%)', 'Recall (%)', 'F1 Score (%)']
x, width   = np.arange(len(metrics)), 0.25

for i, (_, row) in enumerate(results_df.iterrows()):
    axes[0].bar(x + i*width, [row[m] for m in metrics], width,
                label=f"{row['Model']} {row['Architecture']}",
                color=bar_colors[i], edgecolor='black', linewidth=0.6)
axes[0].set_title('All Metrics by Architecture')
axes[0].set_xticks(x + width)
axes[0].set_xticklabels([m.replace(' (%)', '') for m in metrics])
axes[0].set_ylabel('Score (%)'); axes[0].legend(fontsize=8); axes[0].grid(axis='y', alpha=0.3)
axes[0].set_ylim(40, 75)

bars2 = axes[1].bar(results_df['Model'] + '\n' + results_df['Architecture'],
                    results_df['Accuracy (%)'], color=bar_colors,
                    edgecolor='black', linewidth=0.8, width=0.5)
axes[1].set_title('Accuracy Comparison')
axes[1].set_ylabel('Accuracy (%)'); axes[1].grid(axis='y', alpha=0.3)
axes[1].set_ylim(results_df['Accuracy (%)'].min() - 3, 70)
for bar, val in zip(bars2, results_df['Accuracy (%)']):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height()+0.2,
                 f'{val:.2f}%', ha='center', fontweight='bold', fontsize=11)

plt.tight_layout()
plt.savefig('../results/accuracy.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → results/accuracy.png')

---
## Phase 3 — Explainability (Permutation Feature Importance)

In [ ]:
from sklearn.inspection import permutation_importance

best_row   = results_df.loc[results_df['Accuracy (%)'].idxmax()]
best_name  = best_row['Model']
best_model = trained_models[best_name]
print(f'Best model: {best_name} {best_row["Architecture"]} — {best_row["Accuracy (%)"]:.2f}%')

print('Computing permutation importance (10 repeats)...')
t0 = time.time()
perm_imp = permutation_importance(
    best_model, X_test_sc, y_test,
    n_repeats=10, random_state=42, scoring='accuracy', n_jobs=-1
)
print(f'Done in {time.time()-t0:.1f}s')

importance_df = pd.DataFrame({
    'Feature':    feature_names,
    'Importance': perm_imp.importances_mean,
    'Std':        perm_imp.importances_std
}).sort_values('Importance', ascending=False).reset_index(drop=True)

print('\n=== Feature Importance ===')
print(importance_df.round(5).to_string(index=False))

In [ ]:
# ── Feature importance chart ───────────────────────────────────────────────────
from matplotlib.patches import Patch

fig, ax = plt.subplots(figsize=(11, 10))
fi_colors = ['#e74c3c' if v < 0 else '#2ecc71' if v > 0.005 else '#3498db'
             for v in importance_df['Importance'][::-1]]
ax.barh(importance_df['Feature'][::-1], importance_df['Importance'][::-1],
        xerr=importance_df['Std'][::-1], color=fi_colors,
        edgecolor='black', linewidth=0.5,
        error_kw=dict(elinewidth=1.2, capsize=3, ecolor='#555'))
ax.set_title(f'Permutation Feature Importance\nBest Model: {best_name} {best_row["Architecture"]}',
             fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Mean Accuracy Decrease', fontsize=11)
ax.axvline(0, color='black', linewidth=0.8, linestyle='--', alpha=0.5)
ax.legend(handles=[
    Patch(facecolor='#2ecc71', label='High importance (>0.005)'),
    Patch(facecolor='#3498db', label='Low importance (0–0.005)'),
    Patch(facecolor='#e74c3c', label='Negative importance'),
], loc='lower right', fontsize=9)
plt.tight_layout()
plt.savefig('../results/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → results/feature_importance.png')

---
## Dataset Signal Analysis (for Research Paper Discussion)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

majority_baseline = y_test.value_counts(normalize=True).max() * 100
best_acc          = results_df['Accuracy (%)'].max()

print('=== Cross-Algorithm Comparison (same split & scaling) ===')
comparison = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree':       DecisionTreeClassifier(max_depth=10, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    f'MLP {best_row["Architecture"]}': best_model,
}
for name, clf in comparison.items():
    if name != f'MLP {best_row["Architecture"]}':
        clf.fit(X_train_sc, y_train)
    acc = accuracy_score(y_test, clf.predict(X_test_sc))
    print(f'  {name:<30}: {acc*100:.2f}%')

print()
print(f'Majority class baseline    : {majority_baseline:.2f}%')
print(f'Best MLP accuracy          : {best_acc:.2f}%')
print(f'Improvement over random    : +{best_acc - majority_baseline:.2f} pp')
print()
corr_abs = df_model.corr()['placement_status'].drop('placement_status').abs().sort_values(ascending=False)
print(f'Max |feature correlation|  : {corr_abs.max():.4f} ({corr_abs.idxmax()})')
print()
print('Discussion: All algorithms converge to ~57% on this dataset.')
print('This is a characteristic of synthetically generated data where')
print('feature-target correlations are inherently weak (max |r|=0.082).')
print('The MLP still outperforms random guessing and identifies meaningful ranking')
print('of feature importance consistent with domain knowledge (backlogs, internships, coding).')

---
## Save All Artefacts

In [ ]:
import json

# Save model artefacts
joblib.dump(best_model,    '../models/mlp_model.pkl')
joblib.dump(scaler,        '../models/scaler.pkl')
joblib.dump(feature_names, '../models/feature_names.pkl')
joblib.dump(oe,            '../models/ordinal_encoder.pkl')

# Save branch columns list for the app
branch_cols = [c for c in feature_names if c.startswith('branch_')]
joblib.dump(branch_cols, '../models/branch_cols.pkl')

# Classification report
y_pred_best = best_model.predict(X_test_sc)
report_str  = classification_report(y_test, y_pred_best, target_names=['Not Placed', 'Placed'])

with open('../results/classification_report.txt', 'w') as f:
    f.write(f'Best Model  : {best_name} {best_row["Architecture"]}\n')
    f.write('=' * 60 + '\n')
    f.write(report_str)
    f.write('\n\nArchitecture Comparison\n' + '=' * 60 + '\n')
    f.write(results_df.to_string(index=False))
    f.write('\n\nDataset Signal Analysis\n' + '=' * 60 + '\n')
    f.write(f'Majority class baseline  : {majority_baseline:.2f}%\n')
    f.write(f'Best MLP accuracy        : {best_acc:.2f}%\n')
    f.write(f'Improvement over random  : +{best_acc - majority_baseline:.2f} pp\n')
    f.write(f'Max |feature correlation| : {corr_abs.max():.4f} ({corr_abs.idxmax()})\n')
    f.write('\nTop 5 Feature Importances\n' + '=' * 60 + '\n')
    f.write(importance_df.head(5).round(5).to_string(index=False))

print('Saved artefacts:')
for p in ['../models/mlp_model.pkl','../models/scaler.pkl',
          '../models/feature_names.pkl','../models/ordinal_encoder.pkl',
          '../models/branch_cols.pkl','../results/classification_report.txt']:
    print(f'  {p}')

print(f'\n✅ Complete — Best: {best_name} {best_row["Architecture"]} @ {best_acc:.2f}%')